# Deforestation and Recovery Balance — Managed Forest of Quebec

This notebook extracts annual Sentinel-2 composites over the Abitibi-Temiscamingue administrative region (managed public forest, Quebec), detects year-over-year change (dNBR), and computes the net balance between lost and recovering forest cover for the whole study area. It also derives aboveground carbon flux from ESA CCI Biomass.

**Pipeline:** GEE extraction -> annual NDVI/NBR composites -> change detection -> region-wide net balance (ha, per year) -> carbon flux (Mg C, ESA CCI Biomass) -> combined map + chart for the LinkedIn post.

In [ ]:
import sys
sys.path.append('../src')

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

from gee_utils import init_ee, build_annual_stack
from change_detection import build_change_series
from biomass_utils import list_collection_dates, carbon_flux, classify_flux

PROJECT = "your-gee-project-id"
init_ee(PROJECT)

## 1. Study area: Abitibi-Temiscamingue administrative region

Source: "Decoupages administratifs" layer from Donnees Quebec / MRNF (1:1,000,000 scale), filtered to the Abitibi-Temiscamingue region.

In [ ]:
import geopandas as gpd
import glob

shp_candidates = glob.glob("../data/decoupages/**/regio_s.shp", recursive=True)
SHP_PATH = shp_candidates[0]

gdf = gpd.read_file(SHP_PATH)
print("Columns:", list(gdf.columns))
gdf.head()

In [ ]:
# Identify the field holding the region name (check the printed columns above)
REGION_NAME_FIELD = "RES_NM_REG"
print(gdf[REGION_NAME_FIELD].unique())

In [ ]:
gdf_region = gdf[gdf[REGION_NAME_FIELD].str.contains("Abitibi", case=False, na=False)]
gdf_region = gdf_region.to_crs("EPSG:4326")

aoi_fc = geemap.geopandas_to_ee(gdf_region)
AOI = aoi_fc.geometry()

area_ha = AOI.area().divide(10000).getInfo()
print(f"AOI area: {area_ha:,.0f} ha")

Map = geemap.Map()
Map.centerObject(AOI, 8)
Map.addLayer(AOI, {"color": "blue"}, "AOI - Abitibi-Temiscamingue")
Map

## 2. Annual composites (NDVI/NBR)

In [ ]:
YEARS = range(2017, 2026)
annual_stack = build_annual_stack(YEARS, AOI)
print(f"Composites generated: {annual_stack.size().getInfo()}")

## 3. Year-over-year change detection and region-wide net balance

For each consecutive year pair, this computes actual loss and recovery area (ha) across the whole AOI — no spatial subdivision needed for this headline number. Runs one `reduceRegion` per year transition.

In [ ]:
change_pairs = build_change_series(annual_stack, band="NBR")
pixel_area_ha = ee.Image.pixelArea().divide(10000)

records = []
for year, classified in change_pairs:
    loss_ha_img = pixel_area_ha.updateMask(classified.eq(1))
    recovery_ha_img = pixel_area_ha.updateMask(classified.eq(2))

    stats = loss_ha_img.rename("loss_ha").addBands(
        recovery_ha_img.rename("recovery_ha")
    ).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=AOI,
        scale=10,
        maxPixels=1e13,
    ).getInfo()

    loss_ha = stats.get("loss_ha", 0) or 0
    recovery_ha = stats.get("recovery_ha", 0) or 0
    records.append({
        "year": year,
        "loss_ha": loss_ha,
        "recovery_ha": recovery_ha,
        "net_balance_ha": recovery_ha - loss_ha,
    })
    print(f"{year}: loss={loss_ha:,.0f} ha, recovery={recovery_ha:,.0f} ha, net={recovery_ha - loss_ha:,.0f} ha")

balance_df = pd.DataFrame(records)
balance_df["net_balance_cumsum"] = balance_df["net_balance_ha"].cumsum()
balance_df.to_csv("../figures/net_balance_annual.csv", index=False)
balance_df

## 4. Result 1: annual loss vs recovery, cumulative net balance

This is the first chart for the LinkedIn post — real numbers, not a placeholder.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

balance_df.plot(x="year", y=["loss_ha", "recovery_ha"], kind="bar", ax=ax1,
                 color=["firebrick", "forestgreen"])
ax1.set_ylabel("Hectares")
ax1.set_title("Annual loss vs recovery — Abitibi-Temiscamingue")
ax1.legend(["Loss", "Recovery"])

ax2.plot(balance_df["year"], balance_df["net_balance_cumsum"], marker="o", color="darkblue")
ax2.axhline(0, color="gray", linewidth=0.8)
ax2.set_ylabel("Cumulative net balance (ha)")
ax2.set_title("Cumulative net balance (recovery - loss)")

plt.tight_layout()
plt.savefig("../figures/annual_loss_recovery.png", dpi=200)
plt.show()

## 5. Map: latest-year change, with a real legend

Loss in red, recovery in green, stable areas left transparent — with an actual color legend, not a bare tile layer.

In [ ]:
last_year, last_classified = change_pairs[-1]

change_vis = {"min": 0, "max": 2, "palette": ["00000000", "d73027", "1a9850"]}

Map2 = geemap.Map()
Map2.centerObject(AOI, 8)
Map2.addLayer(last_classified, change_vis, f"Change {last_year}")
Map2.add_legend(
    title=f"Cover change ({last_year})",
    legend_dict={"Loss": "d73027", "Recovery": "1a9850"},
)
Map2

## 6. Aboveground carbon flux (ESA CCI Biomass)

ESA CCI Biomass v6.0 provides continuous, gap-free annual aboveground biomass (AGB) maps (2007, 2010, 2015-2022). Comparing two years yields a carbon flux map: positive = sink (net gain), negative = source (net loss). Sentinel-2 covers 2017-2025; ESA CCI Biomass tops out at 2022, so the comparison uses 2017 vs 2022. Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0.

In [ ]:
# Diagnostic: confirm how years are indexed in the collection before trusting
# the calendarRange filter in biomass_utils.get_agb
print(list_collection_dates().getInfo())

In [ ]:
YEAR_T0, YEAR_T1 = 2017, 2022
flux = carbon_flux(YEAR_T0, YEAR_T1, AOI)

flux_stats = flux.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=AOI, scale=100, maxPixels=1e13,
).getInfo()
print(f"Carbon flux {YEAR_T0}-{YEAR_T1} (Mg C/ha): {flux_stats}")

total_change_mg = flux.multiply(pixel_area_ha).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=AOI, scale=100, maxPixels=1e13,
).get("carbon_flux_Mg_ha").getInfo()
print(f"Estimated total carbon stock change {YEAR_T0}-{YEAR_T1}: {total_change_mg:,.0f} Mg C")

In [ ]:
flux_vis = {"min": -20, "max": 20, "palette": ["d73027", "ffffbf", "1a9850"]}

Map3 = geemap.Map()
Map3.centerObject(AOI, 8)
Map3.addLayer(flux, flux_vis, f"Carbon flux {YEAR_T0}-{YEAR_T1} (Mg C/ha)")
Map3.add_colorbar(flux_vis, label="Carbon flux (Mg C/ha): red = source, green = sink")
Map3

## 7. Combined result for LinkedIn

Headline numbers for the post caption, computed directly from the sections above (no placeholders):
- Total area analyzed, cumulative net cover balance ({YEARS.start}-{YEARS.stop-1}).
- Total estimated carbon stock change ({YEAR_T0}-{YEAR_T1}).

In [ ]:
print(f"Study area: {area_ha:,.0f} ha (Abitibi-Temiscamingue)")
print(f"Net cover balance {YEARS.start}-{YEARS.stop - 1}: {balance_df['net_balance_ha'].sum():,.0f} ha "
      f"({'net recovery' if balance_df['net_balance_ha'].sum() >= 0 else 'net loss'})")
print(f"Carbon stock change {YEAR_T0}-{YEAR_T1}: {total_change_mg:,.0f} Mg C "
      f"({'net sink' if total_change_mg >= 0 else 'net source'})")